# WaveSpeedAI Jupyter Studio (RunPod)
**Run cells top-to-bottom.**
- Cell 1 installs dependencies
- Cell 2 prompts for API key and validates it
- Cell 3 defines client, registry, job runner, and UI
- Cell 4 launches the interactive UI


In [ ]:
# Cell 1 - Install dependencies
%pip install httpx==0.27.0 ipywidgets==8.1.2


In [ ]:
# Cell 2 - API key input + validation
from getpass import getpass
import httpx

BASE_URL = "https://api.wavespeed.ai"
MODELS_ENDPOINT = "/api/v3/models"

WAVESPEED_API_KEY = getpass("Enter WaveSpeed API key: ")
if not WAVESPEED_API_KEY:
    raise ValueError("API key is required.")

def _auth_headers():
    return {"Authorization": f"Bearer {WAVESPEED_API_KEY}"}

with httpx.Client(base_url=BASE_URL, headers=_auth_headers(), timeout=30) as client:
    response = client.get(MODELS_ENDPOINT)
    if response.status_code in (401, 403):
        raise PermissionError(f"Auth failed ({response.status_code}): {response.text}")
    response.raise_for_status()
    payload = response.json()
    model_count = len(payload.get("data", payload))
    print(f"API key validated. Models available: {model_count}")


In [ ]:
# Cell 3 - Client, registry, job runner, and notebook UI
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import base64
import json
import threading
import time

import httpx
import ipywidgets as widgets
from IPython.display import display, clear_output

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

@dataclass
class ModelInfo:
    model_id: str
    name: str
    category: str
    api_schema: Dict[str, Any]

class WaveSpeedClient:
    def __init__(self, api_key: str, base_url: str = BASE_URL) -> None:
        self.api_key = api_key
        self.base_url = base_url
        self.client = httpx.Client(
            base_url=self.base_url,
            headers={"Authorization": f"Bearer {self.api_key}"},
            timeout=120,
        )

    def _handle_response(self, response: httpx.Response) -> Dict[str, Any]:
        if response.status_code in (401, 403):
            raise PermissionError(f"Auth failed ({response.status_code}): {response.text}")
        response.raise_for_status()
        return response.json()

    def list_models(self) -> List[Dict[str, Any]]:
        response = self.client.get(MODELS_ENDPOINT)
        payload = self._handle_response(response)
        return payload.get("data", payload)

    def upload_file(self, local_path: str) -> Dict[str, Any]:
        print(f"Uploading file: {local_path}")
        with open(local_path, "rb") as handle:
            files = {"file": handle}
            response = self.client.post("/api/v3/files", files=files)
        return self._handle_response(response)

    def run_model(self, model_id: str, payload: Dict[str, Any]) -> Dict[str, Any]:
        print(f"Submitting job to model: {model_id}")
        response = self.client.post(f"/api/v3/generate/{model_id}", json=payload)
        return self._handle_response(response)

    def poll_job(self, job_id: str) -> Dict[str, Any]:
        response = self.client.get(f"/api/v3/jobs/{job_id}")
        return self._handle_response(response)

    def download_file(self, url: str) -> bytes:
        print(f"Downloading: {url}")
        response = self.client.get(url)
        if response.status_code in (401, 403):
            raise PermissionError(f"Auth failed ({response.status_code}): {response.text}")
        response.raise_for_status()
        return response.content

class ModelRegistry:
    def __init__(self, raw_models: List[Dict[str, Any]]) -> None:
        self.models = self._parse_models(raw_models)

    def _infer_category(self, model: Dict[str, Any]) -> str:
        category = (model.get("type") or model.get("category") or "").lower()
        if category:
            return category
        schema = model.get("api_schema", {})
        props = schema.get("properties", {})
        if "audio" in props or "audio_url" in props:
            return "audio"
        if "video" in props or "video_url" in props:
            if "image" in props or "image_url" in props:
                return "image-to-video"
            return "video"
        if "image" in props or "image_url" in props:
            return "image"
        return "text-to-image"

    def _parse_models(self, raw_models: List[Dict[str, Any]]) -> List[ModelInfo]:
        parsed = []
        for model in raw_models:
            model_id = model.get("id") or model.get("model_id")
            if not model_id:
                continue
            name = model.get("name", model_id)
            api_schema = model.get("api_schema", {}) or {}
            category = self._infer_category(model)
            parsed.append(ModelInfo(model_id=model_id, name=name, category=category, api_schema=api_schema))
        return parsed

    def grouped(self) -> Dict[str, List[ModelInfo]]:
        grouped: Dict[str, List[ModelInfo]] = {}
        for model in self.models:
            grouped.setdefault(model.category, []).append(model)
        return grouped

    def get(self, model_id: str) -> Optional[ModelInfo]:
        for model in self.models:
            if model.model_id == model_id:
                return model
        return None

class JobRunner:
    def __init__(self, client: WaveSpeedClient) -> None:
        self.client = client

    def wait_for_completion(self, job_id: str, output: widgets.Output, poll_interval: float = 3.0, timeout: float = 900) -> Dict[str, Any]:
        start = time.time()
        while True:
            payload = self.client.poll_job(job_id)
            status = payload.get("status")
            progress = payload.get("progress")
            with output:
                print(f"Status: {status} | Progress: {progress}")
            if status in {"succeeded", "failed", "canceled"}:
                return payload
            if time.time() - start > timeout:
                raise TimeoutError("Timed out waiting for job completion")
            time.sleep(poll_interval)

    def download_outputs(self, result: Dict[str, Any], output: widgets.Output) -> List[str]:
        output_urls = result.get("outputs") or result.get("output_urls") or []
        with output:
            print(f"Final output URLs: {output_urls}")
        saved_paths: List[str] = []
        for index, item in enumerate(output_urls, start=1):
            url = item.get("url") if isinstance(item, dict) else item
            if not url:
                continue
            content = self.client.download_file(url)
            suffix = ".bin"
            if isinstance(url, str) and "." in url:
                suffix = "." + url.split(".")[-1].split("?")[0]
            path = OUTPUT_DIR / f"wavespeed_output_{index}{suffix}"
            path.write_bytes(content)
            saved_paths.append(str(path))
            with output:
                print(f"Saved: {path}")
        return saved_paths

class NotebookUI:
    def __init__(self, client: WaveSpeedClient, registry: ModelRegistry) -> None:
        self.client = client
        self.registry = registry
        self.output = widgets.Output()
        self.model_dropdown = widgets.Dropdown(options=self._model_options(), description="Model", layout=widgets.Layout(width="80%"))
        self.inputs_box = widgets.VBox()
        self.submit_button = widgets.Button(description="Run Model", button_style="primary")
        self.schema_view = widgets.Textarea(value="", description="Schema", layout=widgets.Layout(width="80%", height="160px"), disabled=True)
        self.widgets: Dict[str, widgets.Widget] = {}
        self.runner = JobRunner(client)
        self.submit_button.on_click(self._on_submit)
        self.model_dropdown.observe(self._on_model_change, names="value")
        self._build_inputs(self.model_dropdown.value)

    def _model_options(self) -> List[Tuple[str, str]]:
        options = []
        for category, models in sorted(self.registry.grouped().items()):
            for model in models:
                label = f"{category} | {model.model_id} | {model.name}"
                options.append((label, model.model_id))
        return options

    def _schema_properties(self, model: ModelInfo) -> Tuple[Dict[str, Any], List[str]]:
        schema = model.api_schema or {}
        properties = schema.get("properties", {})
        required = schema.get("required", [])
        return properties, required

    def _build_inputs(self, model_id: str) -> None:
        model = self.registry.get(model_id)
        if not model:
            return
        props, required = self._schema_properties(model)
        self.widgets = {}
        rows = []
        for field, schema in props.items():
            label = f"{field} {'*' if field in required else ''}"
            widget = self._widget_for_field(field, schema)
            widget.description = label
            self.widgets[field] = widget
            rows.append(widget)
        self.inputs_box.children = rows
        self.schema_view.value = json.dumps(model.api_schema or {}, indent=2)

    def _widget_for_field(self, field: str, schema: Dict[str, Any]) -> widgets.Widget:
        field_type = schema.get("type", "string")
        if field_type in {"number", "integer"}:
            return widgets.FloatText() if field_type == "number" else widgets.IntText()
        if field_type == "boolean":
            return widgets.Checkbox()
        if field in {"prompt", "negative_prompt"} or "prompt" in field:
            return widgets.Textarea(layout=widgets.Layout(width="80%", height="80px"))
        return widgets.Text(layout=widgets.Layout(width="80%"))

    def _coerce_value(self, field: str, schema: Dict[str, Any], value: Any) -> Any:
        if value in (None, ""):
            return None
        field_type = schema.get("type", "string")
        if field_type == "integer":
            return int(value)
        if field_type == "number":
            return float(value)
        return value

    def _maybe_upload_or_encode(self, schema: Dict[str, Any], value: str) -> Any:
        if not isinstance(value, str):
            return value
        if value.startswith("http://") or value.startswith("https://"):
            return value
        path = Path(value)
        if not path.exists():
            return value
        fmt = (schema.get("format") or "").lower()
        media_type = (schema.get("contentMediaType") or "").lower()
        expects_base64 = fmt in {"base64", "byte"}
        expects_file = fmt in {"binary", "file"} or "image" in media_type or "video" in media_type
        expects_uri = fmt == "uri"
        if expects_base64:
            print(f"Encoding file to base64: {path}")
            return base64.b64encode(path.read_bytes()).decode("utf-8")
        if expects_file or expects_uri:
            upload_payload = self.client.upload_file(str(path))
            return upload_payload.get("url") or upload_payload.get("id") or upload_payload.get("file_id")
        return value

    def _collect_payload(self, model: ModelInfo) -> Dict[str, Any]:
        props, required = self._schema_properties(model)
        payload: Dict[str, Any] = {}
        for field, schema in props.items():
            widget = self.widgets.get(field)
            if widget is None:
                continue
            value = widget.value
            value = self._maybe_upload_or_encode(schema, value)
            value = self._coerce_value(field, schema, value)
            if value is None:
                if field in required:
                    raise ValueError(f"Missing required field: {field}")
                continue
            payload[field] = value
        return payload

    def _on_model_change(self, change: Dict[str, Any]) -> None:
        self._build_inputs(change["new"])

    def _on_submit(self, _btn: widgets.Button) -> None:
        with self.output:
            clear_output()
            print("Submitting job...")
        model = self.registry.get(self.model_dropdown.value)
        if not model:
            with self.output:
                print("No model selected.")
            return
        try:
            payload = self._collect_payload(model)
        except Exception as exc:
            with self.output:
                print(f"Input error: {exc}")
            return

        def _run_job():
            job_response = self.client.run_model(model.model_id, payload)
            job_id = job_response.get("job_id") or job_response.get("id")
            if not job_id:
                with self.output:
                    print(f"Job submission failed: {job_response}")
                return
            result = self.runner.wait_for_completion(job_id, self.output)
            if result.get("status") != "succeeded":
                with self.output:
                    print(f"Job failed: {result}")
                return
            self.runner.download_outputs(result, self.output)

        thread = threading.Thread(target=_run_job, daemon=True)
        thread.start()

    def show(self) -> None:
        display(widgets.HTML("<h3>WaveSpeedAI Notebook Studio</h3>"))
        display(self.model_dropdown)
        display(self.inputs_box)
        display(self.submit_button)
        display(self.schema_view)
        display(self.output)

client = WaveSpeedClient(WAVESPEED_API_KEY)
registry = ModelRegistry(client.list_models())
ui = NotebookUI(client, registry)


In [ ]:
# Cell 4 - Launch UI
ui.show()
